# Release notes
Paths and language were normalized for this source release. Historical training protocols remain documented in `REVIEW-1023.md`; retrain after any protocol corrections.


In [ ]:
from pathlib import Path
import sys
import copy
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "models/DeepLearnModel.py").is_file() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the MMHRP-CAMoPE-1023 project.")
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install rdkit==2024.9.5
# !pip install torch_geometric==2.5.3

In [ ]:
import os, sys

# Inspect the current working directory
print("Current working directory:", os.getcwd())

# Inspect the Python module search path
print("Python module search path:")
for p in sys.path:
    print(p)


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
import sys
import torch
from torch.utils.data import DataLoader
# doc_name = str(PROJECT_ROOT / '')
doc_name = str(PROJECT_ROOT)
sys.path.append(doc_name)
from utils.rxn import *
from utils.molecule import *
from torch_geometric.loader import DataLoader
from models.DeepLearnModel import *
import time
from tqdm import tqdm
import datetime
import warnings
warnings.simplefilter('ignore')

In [ ]:
# rs_list = [1,2,3,4,5]
rs_list = [42]
eval_metrics = np.zeros((1+len(rs_list), 6))
columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
index = []
for rs in rs_list:
  index.append("%s" % rs)
index.append("avg±std")
eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

In [ ]:
for rs in rs_list:

    # =====================================================
    # 1. Import data
    # =====================================================
    data = pd.read_excel("%s/data/AHO/2AHO_all_condition.xlsx" % doc_name)

    random_state = rs
    data = data.sample(random_state=random_state, frac=1).reset_index(drop=True)

    vocab_path = "%s/models/AHO_vocab.txt" % doc_name

    # =====================================================
    # 2. Build dataset
    # =====================================================

    rxn_list = df_to_AHO_rxn_list(data)

    rxn_RxnSmi = []
    max_len = -1

    for rxn in rxn_list:
        RxnSmi = get_AHO_RxnSmi(rxn)
        max_len = max(max_len, len(RxnSmi))
        RxnSmi = " ".join(smi_tokenizer(RxnSmi))
        rxn_RxnSmi.append(RxnSmi)

    rxn_dataset = []
    smi_inputsize = 128

    for batch in tqdm(range(len(rxn_list))):

        rxn = rxn_list[batch]
        meta = []

        reac = rxn.reactants
        prod = rxn.products

        meta.append(
            smis_to_graph(reac + prod, ChargeContribution=False)
        )

        sols = rxn.solvents
        cats = rxn.cats

        meta.append(
            smis_to_graph(sols + cats, ChargeContribution=False)
        )

        RxnSmi_vec = RxnSmi_to_tensor(
            RxnSmi=rxn_RxnSmi[batch],
            maxlen_=max_len,
            victor_size=smi_inputsize,
            file=vocab_path
        )

        meta.append(RxnSmi_vec)

        cond_feat = torch.tensor(
            [rxn.temp, rxn.pres, rxn.s],
            dtype=torch.float32
        )

        meta.append(cond_feat)

        meta.append(rxn.rxn_ddg)

        rxn_dataset.append(meta)

    # =====================================================
    # Report directory
    # =====================================================

    base_dir = str(PROJECT_ROOT / 'exp/AHO')

    dir_path = os.path.join(
        base_dir,
        f"AHO_MMHRP-GCL-tri-MoPE-tidu_rs=42"
    )

    os.makedirs(dir_path, exist_ok=True)

    report_path = os.path.join(
        dir_path,
        "Model_Training_Report.txt"
    )

    f = open(report_path, "w")

    # =====================================================
    # Train / Test split
    # =====================================================

    ratio = 0.8
    batch_size = 256

    batch = len(rxn_dataset)
    train_set = rxn_dataset[0: int(ratio * batch)]
    test_set = rxn_dataset[int(ratio * batch):]

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=18, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=18, pin_memory=True, persistent_workers=True)

    # =====================================================
    # Metrics
    # =====================================================

    test_RMSE = []
    test_R2 = []
    test_MAE = []

    train_R2 = []
    train_RMSE = []
    train_MAE = []

    # =====================================================
    # 3. Model
    # =====================================================

    t = 2000 #2000
    lr = 1e-4
    num_feature = 7
    smi_inputsize = 128
    Cond_OutDim = 32

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = MMHRP_GCL_tri_MoPE_pdall(

        GraphEncoder={
            "NodeFeatNum": num_feature,
            "Channels": [32, 64],
            "Heads": 4
        },

        TextEncoder={
            "SmiFeatNum": smi_inputsize,
            "Heads": 4,
            "BigruChannels": [128, 128],
            "BigruNumlayer": 1
        },

        ConditionEncoder={
            "InputDim": 3,
            "HiddenDim": 64,
            "OutDim": Cond_OutDim
        },

        Decoder={
            "Channels": [1000, 500, 100]
        },

        emb_size=128,
        device=device,
        FDS=None
    )

    model = model.to(device)

    opti = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-5
    )

    criterion = nn.MSELoss()

    # =====================================================
    # best record
    # =====================================================

    best = [0,0,0,0,0,0,[],[],None]

    # =====================================================
    # 4. Training
    # =====================================================

    for epoch in tqdm(range(t)):

        model.train()
        global_loss = 0

        for data in train_loader:

            x = [i.to(device) for i in data[:-1]]

            y = torch.unsqueeze(
                data[-1],
                dim=1
            ).float().to(device)

            pred = model.forward(
                x,
                y=y,
                epoch=epoch,
                device=device
            )

            # ===== NaN prediction detection =====
            if torch.isnan(pred).any():
                print(f"NaN prediction at epoch {epoch}, skip batch")
                continue

            loss = criterion(pred, y)

            # ===== NaN loss detection =====
            if torch.isnan(loss):
                print(f"NaN loss at epoch {epoch}, skip batch")
                continue

            opti.zero_grad()

            loss.backward()

            # ===== Gradient clipping =====
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)

            # ===== Gradient NaN detection =====
            skip = False
            for p in model.parameters():
                if p.grad is not None and torch.isnan(p.grad).any():
                    print(f"NaN gradient at epoch {epoch}, skip batch")
                    skip = True
                    break

            if skip:
                opti.zero_grad()
                continue

            opti.step()

            global_loss += loss.item()

        # =====================================================
        # Train evaluation
        # =====================================================

        model.eval()

        with torch.no_grad():

            pred = []
            true = []

            for data in train_loader:

                x = [i.to(device) for i in data[:-1]]

                tr = list(
                    torch.unsqueeze(data[-1], dim=1).numpy()
                )

                pr = list(
                    model.forward(x, device=device)
                    .cpu()
                    .numpy()
                )

                pred += pr
                true += tr

            pred_arr = np.array(pred).flatten()
            true_arr = np.array(true).flatten()

            mask = ~np.isnan(pred_arr)

            pred_arr = pred_arr[mask]
            true_arr = true_arr[mask]

            train_RMSE.append(RMSE(pred_arr, true_arr))
            train_R2.append(R2(pred_arr, true_arr))
            train_MAE.append(MAE(true_arr, pred_arr))

        # =====================================================
        # Test evaluation
        # =====================================================

        with torch.no_grad():

            pred = []
            true = []

            for data in test_loader:

                x = [i.to(device) for i in data[:-1]]

                tr = list(
                    torch.unsqueeze(data[-1], dim=1).numpy()
                )

                pr = list(
                    model.forward(x, device=device)
                    .cpu()
                    .numpy()
                )

                pred += pr
                true += tr

            pred_arr = np.array(pred).flatten()
            true_arr = np.array(true).flatten()

            mask = ~np.isnan(pred_arr)

            pred_arr = pred_arr[mask]
            true_arr = true_arr[mask]

            test_RMSE.append(RMSE(pred_arr, true_arr))
            test_R2.append(R2(pred_arr, true_arr))
            test_MAE.append(MAE(true_arr, pred_arr))

            if epoch == 0 or test_R2[-1] > best[3]:

                best = [
                    train_R2[-1],
                    train_RMSE[-1],
                    train_MAE[-1],
                    test_R2[-1],
                    test_RMSE[-1],
                    test_MAE[-1],
                    pred,
                    true,
                    copy.deepcopy(model.state_dict())
                ]

        print(
            "Epoch:%d loss:%f R2(train)=%.3f R2(test)=%.3f"
            % (
                epoch + 1,
                global_loss / len(train_loader),
                train_R2[-1],
                test_R2[-1],
            )
        )

    # Evaluation
    f.write("\n")
    # Performance in train set
    f.write("R2 of train set is:%.3f+-%f\tbest:%f\n" % (
    np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
    f.write("RMSE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))
    f.write("MAE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_MAE[-10:]).mean(), np.array(train_MAE[-10:]).std(), best[2]))

    # Performance in test set
    f.write("R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (
    np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
    f.write("RMSE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))
    f.write("MAE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_MAE[-10:]).mean(), np.array(test_MAE[-10:]).std(), best[5]))

    f.close()

    # Performance in train set
    print("R2 of train set is:%.3f+-%f\tbest:%f\n" % (
    np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
    print("RMSE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))
    print("MAE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_MAE[-10:]).mean(), np.array(train_MAE[-10:]).std(), best[2]))

    # Performance in test set
    print("R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (
    np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
    print("RMSE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))
    print("MAE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_MAE[-10:]).mean(), np.array(test_MAE[-10:]).std(), best[5]))

    # =====================================================
    # Save metrics
    # =====================================================

    eval_metrics.loc["%s" % rs]["train_R2"] = best[0]
    eval_metrics.loc["%s" % rs]["train_RMSE"] = best[1]
    eval_metrics.loc["%s" % rs]["train_MAE"] = best[2]
    eval_metrics.loc["%s" % rs]["test_R2"] = best[3]
    eval_metrics.loc["%s" % rs]["test_RMSE"] = best[4]
    eval_metrics.loc["%s" % rs]["test_MAE"] = best[5]
    

    # 5.Figure
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig = plt.figure(dpi=300, figsize=(10, 5))

    # Training Fig
    plt.subplot(1, 2, 1)
    steps = np.linspace(1, t, t)
    plt.plot(steps, train_R2, color=[236 / 255, 164 / 255, 124 / 255])
    plt.plot(steps, test_R2, color=[117 / 255, 157 / 255, 219 / 255])
    # Beautify
    plt.legend(["train set", "test set"], loc="upper left", prop={'size': 8})
    plt.xlabel("Epoch", fontsize=10)
    plt.ylabel("R$^2$", fontsize=10)
    plt.title("The R$^2$ of train & test set during training", fontsize=13)

    # Test set performance
    plt.subplot(1, 2, 2)
    tr = np.array(best[7]).flatten() * 100
    pr = np.array(best[6]).flatten() * 100
    plt.scatter(pr, tr, alpha=0.7, marker=".")
    plt.xlabel("Predicted ΔΔG (kcal mol$^-$$^1$)", fontsize=10)
    plt.ylabel("Observed YΔΔG (kcal mol$^-$$^1$)", fontsize=10)
    x = np.linspace(0, 100, 100)
    y = np.linspace(0, 100, 100)
    plt.plot(x, y, linestyle="--", color="r")
    plt.title("Test set performance", fontsize=15)

    fig.suptitle("MMHRP-GCL for AHO dataset", fontsize=16)
    plt.tight_layout()
    plt.savefig("%s/Performance_Figure.png" % dir_path)
    plt.show()

    torch.save(best[8], "%s/model.pth" % dir_path)

In [ ]:
# Evaluation metrics report
for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
  for j in range(eval_metrics.shape[1]):
    eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
# eval_metrics.to_csv("%s/exp/Heck_Benchmark/MMHRP-GCL_report_%s.csv" % (doc_name, datetime.datetime.now()))
eval_metrics.to_csv(str(PROJECT_ROOT / 'exp/AHO/AHO_MMHRP-GCL-tri-MoPE-tidu_rs=42/AHO_MMHRP-GCL-tri-MOPE_report-tidu_rs=42.csv') )#% (doc_name, datetime.datetime.now()
print(eval_metrics)